# Day 1 — Exploratory Data Analysis
## Healthcare Fraud Detection
**UGASS DSC x Zindi Capstone**

---
## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/healthcare_fraud_detection.csv')
print('Shape:', df.shape)
df.head()

---
## 2. Basic Info & Summary Stats

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe().round(2)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Columns with missing values:')
print(missing)
print(f'\nTotal missing: {missing.sum()} rows ({missing.sum()/len(df)*100:.1f}% of dataset)')

---
## 3. Plot 1 — Target Distribution (Class Imbalance)
> **Why:** The most critical check. If fraud is rare, accuracy alone is a useless metric.

In [ ]:
fraud_counts = df['Is_Fraud'].value_counts()
fraud_pct = df['Is_Fraud'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Not Fraud (0)', 'Fraud (1)'], fraud_counts, color=['#4C72B0', '#DD8452'])
axes[0].set_title('Fraud vs Not Fraud — Count')
axes[0].set_ylabel('Number of Claims')
for i, (count, pct) in enumerate(zip(fraud_counts, fraud_pct)):
    axes[0].text(i, count + 80, f'{count}\n({pct:.1f}%)', ha='center', fontsize=10)

# Pie chart
axes[1].pie(fraud_counts, labels=['Not Fraud', 'Fraud'], autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Fraud Proportion')

plt.suptitle('Class Distribution — Is_Fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot1_class_distribution.png', bbox_inches='tight')
plt.show()

print('\n⚠ IMBALANCED DATASET: Only 8.3% of claims are fraudulent.')
print('→ Do NOT rely on accuracy. Use F1-Score and ROC-AUC instead.')
print('→ Use class_weight="balanced" or SMOTE during modeling.')

---
## 4. Plot 2 — Claim Amount Distribution by Fraud Label
> **Why:** Fraudulent claims tend to have higher amounts — this is often your strongest signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KDE / histogram
for fraud_label, color, label in [(0, '#4C72B0', 'Not Fraud'), (1, '#DD8452', 'Fraud')]:
    subset = df[df['Is_Fraud'] == fraud_label]['Claim_Amount']
    axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Claim Amount Distribution')
axes[0].set_xlabel('Claim Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
df_plot = df[['Claim_Amount', 'Is_Fraud']].copy()
df_plot['Fraud Label'] = df_plot['Is_Fraud'].map({0: 'Not Fraud', 1: 'Fraud'})
axes[1].boxplot([df[df['Is_Fraud']==0]['Claim_Amount'], df[df['Is_Fraud']==1]['Claim_Amount']],
                labels=['Not Fraud', 'Fraud'], patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.6))
axes[1].set_title('Claim Amount — Boxplot by Fraud Label')
axes[1].set_ylabel('Claim Amount ($)')

plt.suptitle('Claim Amount vs Fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot2_claim_amount.png', bbox_inches='tight')
plt.show()

print('Mean Claim Amount:')
print(df.groupby('Is_Fraud')['Claim_Amount'].mean().rename({0: 'Not Fraud', 1: 'Fraud'}))
print('\n→ Fraudulent claims average nearly 2x higher than legitimate claims.')

---
## 5. Plot 3 — Correlation Matrix
> **Why:** Reveals which numerical features are related to each other and to the target (Is_Fraud).

In [ ]:
numerical_cols = ['Patient_Age', 'Claim_Amount', 'Approved_Amount',
                  'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly',
                  'Length_of_Stay', 'Prior_Visits_12m', 'Chronic_Condition_Flag', 'Is_Fraud']

corr_matrix = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot3_correlation_matrix.png', bbox_inches='tight')
plt.show()

# Print top correlations with Is_Fraud
fraud_corr = corr_matrix['Is_Fraud'].drop('Is_Fraud').sort_values(key=abs, ascending=False)
print('Feature correlations with Is_Fraud (sorted by strength):')
print(fraud_corr.round(3))

---
## 6. Plot 4 — Fraud Rate by Categorical Features
> **Why:** Shows which categories are associated with higher fraud rates — key for feature engineering.

In [ ]:
cat_cols = ['Visit_Type', 'Insurance_Type', 'Provider_Specialty', 'Patient_Gender']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    fraud_rate = df.groupby(col)['Is_Fraud'].mean().sort_values(ascending=False) * 100
    bars = axes[i].bar(fraud_rate.index, fraud_rate.values, color='#4C72B0', alpha=0.8)
    axes[i].set_title(f'Fraud Rate by {col}')
    axes[i].set_ylabel('Fraud Rate (%)')
    axes[i].set_ylim(0, fraud_rate.max() * 1.3)
    axes[i].tick_params(axis='x', rotation=30)
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.suptitle('Fraud Rate by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot4_fraud_by_category.png', bbox_inches='tight')
plt.show()

---
## 7. Plot 5 — Provider-Level Analysis
> **Why:** Fraud patterns often cluster around specific providers — a strong signal.

In [ ]:
provider_stats = df.groupby('Provider_ID').agg(
    total_claims=('Claim_ID', 'count'),
    fraud_rate=('Is_Fraud', 'mean'),
    avg_claim=('Claim_Amount', 'mean')
).reset_index()
provider_stats['fraud_rate_pct'] = provider_stats['fraud_rate'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: claims volume vs fraud rate
axes[0].scatter(provider_stats['total_claims'], provider_stats['fraud_rate_pct'],
               alpha=0.5, color='#4C72B0', s=40)
axes[0].set_xlabel('Total Claims per Provider')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Provider: Claim Volume vs Fraud Rate')

# Top 15 highest-fraud providers
top_fraud = provider_stats.nlargest(15, 'fraud_rate_pct')
axes[1].barh(top_fraud['Provider_ID'], top_fraud['fraud_rate_pct'], color='#DD8452', alpha=0.8)
axes[1].set_xlabel('Fraud Rate (%)')
axes[1].set_title('Top 15 Providers by Fraud Rate')

plt.suptitle('Provider-Level Fraud Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/plot5_provider_analysis.png', bbox_inches='tight')
plt.show()

print(f'Providers with >20% fraud rate: {(provider_stats["fraud_rate"] > 0.2).sum()}')
print('→ Provider_ID fraud rate will be a powerful engineered feature tomorrow.')

---
## 8. Day 1 Summary & Key Findings

In [ ]:
print('=== DAY 1 EDA SUMMARY ===')
print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {df["Is_Fraud"].mean()*100:.1f}%  ← IMBALANCED')
print(f'Missing values: Insurance_Type, Provider_Specialty, Prior_Visits_12m (350 each)')
print()
print('Top findings:')
print('  1. Fraudulent claims are ~2x higher in Claim_Amount')
print('  2. Claim_Amount is the strongest predictor of fraud')
print('  3. Provider-level fraud rates vary significantly')
print()
print('Tomorrow (Day 2):')
print('  - Fill missing values (mode for categoricals, median for numeric)')
print('  - One-Hot Encode: Insurance_Type, Visit_Type, Provider_Specialty, Patient_Gender')
print('  - Engineer features: provider_fraud_rate, claim_to_approved_ratio')
print('  - Scale: Claim_Amount, Approved_Amount, Prior_Visits_12m')